In [56]:
# Gurobi
import gurobipy as gp
import pickle
from gurobipy import GRB
import pandas as pd
import numpy as np
with open('../data/cs_metrics.pkl', 'rb') as f:
    cs_metrics = pickle.load(f)
# cs_configのロード
cs_config = pd.read_csv('../data/cs_config.csv', index_col=0)


In [57]:
# データの確認
print(cs_config.head())
cs_metrics

        ports  cap_kw  total_ports
csids                             
900000      2      90           11
900002      2      90           11
900004      2      90           11
900006      0      90           11
900008      1      90           11


,利用率,設置容量,初期コスト,充足率,充電損失量,待ち発生率,待ち時間平均,迂回距離平均
CSID,,,,,,,,
90000000,0.392708,[180],1692,88.696758,216.19674,0.071429,30.407143,358.493114
90000200,0.167014,[180],1692,100.000000,0.00000,0.000000,0.000000,1015.182899
90000400,0.152083,[180],1692,100.000000,0.00000,0.166667,31.933333,2313.674383
90000800,0.140972,[90],846,100.000000,0.00000,0.000000,0.000000,70.544940
90001000,0.000000,[90],846,100.000000,0.00000,0.000000,0.000000,0.000000
90001200,0.000000,[180],1692,100.000000,0.00000,0.000000,0.000000,0.000000
90001600,0.000000,[90],846,100.000000,0.00000,0.000000,0.000000,0.000000


In [58]:
# ✅ 決定変数の設計パターン

# パターン1: 選択問題（バイナリ）
# 出力: どのCSを選ぶか？
model1 = gp.Model("CS_Selection")
cs_indices = range(10)
x = model1.addVars(cs_indices, vtype=GRB.BINARY, name="select_cs")
# 出力例: x[0]=1, x[1]=0, x[2]=1, ... → CS0とCS2を選択

# パターン2: 容量決定問題（整数）
# 出力: 各CSに何台の充電器を設置するか？
model2 = gp.Model("CS_Capacity")
y = model2.addVars(cs_indices, vtype=GRB.INTEGER, lb=0, ub=5, name="num_chargers")
# 出力例: y[0]=3, y[1]=0, y[2]=2, ... → CS0に3台、CS2に2台

# パターン3: 連続値決定問題
# 出力: 各CSの運用レベルは？
model3 = gp.Model("CS_Operation")
z = model3.addVars(cs_indices, vtype=GRB.CONTINUOUS, lb=0, ub=1, name="operation_level")
# 出力例: z[0]=0.8, z[1]=0.0, z[2]=0.6, ... → CS0は80%稼働

print("決定変数のタイプにより出力形式が決まる")


決定変数のタイプにより出力形式が決まる


In [59]:
# ✅ OptunaとGurobiの比較
import optuna

print("=== OptunaとGurobiの違い ===")
comparison = {
    'Gurobi': {
        '問題種類': '数理最適化（MILP、LP、QP等）',
        '解法': '厳密解法（分枝限定法、シンプレックス法）',
        '出力': '最適解（保証あり）',
        '適用': '制約付き最適化、組み合わせ最適化',
        '変数': '決定変数（連続、整数、バイナリ）'
    },
    'Optuna': {
        '問題種類': 'ハイパーパラメータ最適化',
        '解法': '試行錯誤型（ベイズ最適化、進化的アルゴリズム）',
        '出力': '最良解（探索結果）',
        '適用': '機械学習、深層学習、シミュレーション',
        '変数': 'ハイパーパラメータ（学習率、層数、etc）'
    }
}

for lib, features in comparison.items():
    print(f"\n{lib}:")
    for key, value in features.items():
        print(f"  {key}: {value}")

=== OptunaとGurobiの違い ===

Gurobi:
  問題種類: 数理最適化（MILP、LP、QP等）
  解法: 厳密解法（分枝限定法、シンプレックス法）
  出力: 最適解（保証あり）
  適用: 制約付き最適化、組み合わせ最適化
  変数: 決定変数（連続、整数、バイナリ）

Optuna:
  問題種類: ハイパーパラメータ最適化
  解法: 試行錯誤型（ベイズ最適化、進化的アルゴリズム）
  出力: 最良解（探索結果）
  適用: 機械学習、深層学習、シミュレーション
  変数: ハイパーパラメータ（学習率、層数、etc）


In [60]:

# データ読み込み確認
print(f"CS候補地数: {len(cs_config)}")
print(f"利用可能な指標: {list(cs_metrics.columns)}")

CS候補地数: 9
利用可能な指標: ['利用率', '設置容量', '初期コスト', '充足率', '充電損失量', '待ち発生率', '待ち時間平均', '迂回距離平均']


In [72]:
# ✅ 充電ステーション口数・出力同時最適化

print("=== 充電ステーション口数・出力最適化問題 ===")

# データ読み込み確認
print(f"CS候補地数: {len(cs_config)}")
print(f"利用可能な指標: {list(cs_metrics.columns)}")

# 最適化モデル作成
model = gp.Model("CS_Port_Power_Optimization")

# インデックス設定
num_candidate_cs = len(cs_config)
cs_indices = range(num_candidate_cs) 
port_options = [0, 1, 2, 3, 4]  # 口数選択肢
power_options = [50, 100]       # 定格出力選択肢 (kW)

print(f"候補地数: {num_candidate_cs}")
print(f"口数選択肢: {port_options}")
print(f"出力選択肢: {power_options}")

# 決定変数の定義
# x[i,p,w]: CS i で口数 p、出力 w を選択するかどうか
x = model.addVars(cs_indices, port_options, power_options, 
                  vtype=GRB.BINARY, name="select")

# 現在の決定変数の確認

print("決定変数 x[i,p,w] を定義完了")
print("  i: CS候補地インデックス")
print("  p: 口数 (0,1,2,3,4)")
print("  w: 定格出力 (50,100 kW)")

=== 充電ステーション口数・出力最適化問題 ===
CS候補地数: 9
利用可能な指標: ['利用率', '設置容量', '初期コスト', '充足率', '充電損失量', '待ち発生率', '待ち時間平均', '迂回距離平均']
候補地数: 9
口数選択肢: [0, 1, 2, 3, 4]
出力選択肢: [50, 100]
決定変数 x[i,p,w] を定義完了
  i: CS候補地インデックス
  p: 口数 (0,1,2,3,4)
  w: 定格出力 (50,100 kW)


In [73]:
# ✅ 制約条件の設定
print("\n=== 制約条件の設定 ===")

# 制約1: 各CSで最大1つの組み合わせのみ選択
for i in cs_indices:
    model.addConstr(
        gp.quicksum(x[i,p,w] for p in port_options for w in power_options) <= 1,
        f"unique_selection_{i}"
    )

# 制約2: 口数が0の場合、出力はどの値でもよい（論理制約）
# 口数が0でない場合のみ、出力が意味を持つ
for i in cs_indices:
    # 口数0の場合の制約: 50kWと100kWのどちらを選んでも同等
    # （この制約により、口数0の場合は自動的に出力が決定される）
    model.addConstr(
        x[i,0,50] + x[i,0,100] <= 1,
        f"port_zero_constraint_{i}"
    )

# 制約3: 総設置箇所数の制限
min_cs = 1
max_cs = num_candidate_cs # 設置可能な最大CS数
model.addConstr(
    gp.quicksum(x[i,p,w] for i in cs_indices for p in port_options for w in power_options if p > 0) >= min_cs,
    "min_cs_count"
)
model.addConstr(
    gp.quicksum(x[i,p,w] for i in cs_indices for p in port_options for w in power_options if p > 0) <= max_cs,
    "max_cs_count"
)

# 総充電出力の上限
MAX_TOTAL_POWER = 500  # 最大総出力 (kW)
model.addConstr(
    gp.quicksum(p * w * x[i,p,w] for i in cs_indices for p in port_options for w in power_options if p > 0) <= MAX_TOTAL_POWER,
    "max_total_power"
)


# 予算制約　出力に応じたコスト設定
BUDGET = 10000  # 総予算 (円)
# 充電器のコスト設定（万円）
CHARGER_COSTS = {50: 380, 100: 730}  # 定格出力別単価
INSTALLATION_COST = 200  # 設置基本コスト（万円）
# コスト計算
def calculate_total_cost():
    return gp.quicksum(
        (CHARGER_COSTS[w] * p + INSTALLATION_COST) * x[i,p,w]
        for i in cs_indices for p in port_options for w in power_options if p > 0
    )

model.addConstr(calculate_total_cost() <= BUDGET, "budget_constraint")

print(f"制約1: 各CSで最大1つの組み合わせのみ選択")
print(f"制約2: 口数0の場合の論理制約")
print(f"制約3: 設置CS数は{min_cs}〜{max_cs}箇所")
print(f"制約4: 最大総出力は{MAX_TOTAL_POWER} kW")
print(f"制約5: 総予算は{BUDGET}万円")




=== 制約条件の設定 ===
制約1: 各CSで最大1つの組み合わせのみ選択
制約2: 口数0の場合の論理制約
制約3: 設置CS数は1〜9箇所
制約4: 最大総出力は500 kW
制約5: 総予算は10000万円


In [63]:
cs_indices = range(len(cs_metrics))

In [74]:
# 目的関数の設定
print("\n=== 目的関数の設定 ===")
# 目的関数: 充足率最大かつ，利用率最大化。この二つの指標は負の相関があることに留意


# ✅ 推奨：Min-Max正規化
def normalize_objective_function():
    """目的関数の正規化"""
    
    # Min-Max正規化（0-1スケール）
    satisfaction_min = cs_metrics['充足率'].min()
    satisfaction_max = cs_metrics['充足率'].max()
    utilization_min = cs_metrics['利用率'].min()
    utilization_max = cs_metrics['利用率'].max()
    
    print(f"充足率範囲: {satisfaction_min:.2f} - {satisfaction_max:.2f}")
    print(f"利用率範囲: {utilization_min:.2f} - {utilization_max:.2f}")
    
    # 正規化された目的関数
    model.setObjective(
        gp.quicksum(
            ((cs_metrics['充足率'][i] - satisfaction_min) / (satisfaction_max - satisfaction_min)) * x[i,p,w] 
            for i in cs_indices for p in port_options for w in power_options if p > 0
        ) +
        gp.quicksum(
            ((cs_metrics['利用率'][i] - utilization_min) / (utilization_max - utilization_min)) * x[i,p,w] 
            for i in cs_indices for p in port_options for w in power_options if p > 0
        ),
        GRB.MAXIMIZE
    )
    
    return True

# 正規化を適用
normalize_objective_function()
print("✅ スケール問題を解決した目的関数に更新")


=== 目的関数の設定 ===
充足率範囲: 88.70 - 100.00
利用率範囲: 0.00 - 0.39


C:\Users\echiz\AppData\Local\Temp\ipykernel_1836\3319704584.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ((cs_metrics['充足率'][i] - satisfaction_min) / (satisfaction_max - satisfaction_min)) * x[i,p,w]


IndexError: index 7 is out of bounds for axis 0 with size 7

In [65]:
# 最適化
model.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 22 rows, 90 columns and 396 nonzeros
Model fingerprint: 0x5c24c058
Variable types: 0 continuous, 90 integer (90 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+04]
Found heuristic solution: objective 5.1715296
Presolve removed 22 rows and 90 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 32 available processors)

Solution count 2: 6.17153 5.17153 

Optimal solution found (tolerance 1.00e-04)
Best objective 6.171529619805e+00, best bound 6.171529619805e+00, gap 0.0000%


In [71]:
model.status

2

In [66]:

# 結果の表示
for i in cs_indices:
    for p in port_options:
        for w in power_options:
            if x[i,p,w].x > 0:
                print(f"CS {i}: 口数 {p}, 出力 {w} kW")
# 最適化結果の確認
print("\n=== 最適化結果 ===")
if model.status == GRB.OPTIMAL:
    print("最適解が見つかりました。")
    for v in model.getVars():
        if v.x > 0:
            print(f"{v.varName}: {v.x}")
else:
    print("最適解が見つかりませんでした。ステータス:", model.status)
# モデルの保存
model.write("cs_port_power_optimization.lp")
# モデルの詳細情報
print("\n=== モデル詳細情報 ===")
print(f"目的関数値: {model.objVal}")
print(f"変数数: {model.numVars}")
print(f"制約数: {model.numConstrs}")
# 出力の確認
print("\n=== 出力の確認 ===")


CS 1: 口数 1, 出力 50 kW
CS 2: 口数 1, 出力 50 kW
CS 3: 口数 1, 出力 50 kW
CS 4: 口数 1, 出力 50 kW
CS 5: 口数 1, 出力 50 kW

=== 最適化結果 ===
最適解が見つかりました。
select[1,1,50]: 1.0
select[2,1,50]: 1.0
select[3,1,50]: 1.0
select[4,1,50]: 1.0
select[5,1,50]: 1.0

=== モデル詳細情報 ===
目的関数値: 6.171529619805482
変数数: 90
制約数: 22

=== 出力の確認 ===


In [67]:
# 目的関数を確認
print("目的関数の詳細分析:")
selected_cs = [1, 2, 3, 4, 5]

total_objective = 0
for cs_idx in selected_cs:
    satisfaction = cs_metrics['充足率'][cs_idx]
    utilization = cs_metrics['利用率'][cs_idx]
    contribution = satisfaction + utilization
    total_objective += contribution
    
    print(f"CS{cs_idx}: 充足率{satisfaction:.2f} + 利用率{utilization:.2f} = {contribution:.2f}")

print(f"総目的関数値: {total_objective:.2f}")
print(f"実際の目的関数値: 500.46")

目的関数の詳細分析:
CS1: 充足率100.00 + 利用率0.17 = 100.17
CS2: 充足率100.00 + 利用率0.15 = 100.15
CS3: 充足率100.00 + 利用率0.14 = 100.14
CS4: 充足率100.00 + 利用率0.00 = 100.00
CS5: 充足率100.00 + 利用率0.00 = 100.00
総目的関数値: 500.46
実際の目的関数値: 500.46


C:\Users\echiz\AppData\Local\Temp\ipykernel_1836\996145718.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  satisfaction = cs_metrics['充足率'][cs_idx]
C:\Users\echiz\AppData\Local\Temp\ipykernel_1836\996145718.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  utilization = cs_metrics['利用率'][cs_idx]


In [68]:
# ✅ 現在のコスト構造による影響分析
print("=== コスト構造による影響分析 ===")

# 充電器のコスト設定（万円）
CHARGER_COSTS = {50: 380, 100: 730}  # 定格出力別単価
INSTALLATION_COST = 200  # 設置基本コスト（万円）

# コスト効率の比較
print("コスト効率分析:")
for power in [50, 100]:
    cost_per_kw = CHARGER_COSTS[power] / power
    print(f"{power}kW充電器: {CHARGER_COSTS[power]}万円, コスト効率: {cost_per_kw:.2f}万円/kW")

print(f"\n100kW充電器は50kW充電器の{CHARGER_COSTS[100]/CHARGER_COSTS[50]:.2f}倍のコスト")
print(f"しかし出力は{100/50:.1f}倍のため、コスト効率は{(CHARGER_COSTS[50]/50)/(CHARGER_COSTS[100]/100):.2f}倍良い")

# 予算15000万円での各パターンの設置可能数
budget = 15000
print(f"\n予算{budget}万円での設置可能数:")
for power in [50, 100]:
    for ports in [1, 2, 3, 4]:
        total_cost = CHARGER_COSTS[power] * ports + INSTALLATION_COST
        max_stations = budget // total_cost
        total_capacity = max_stations * ports * power
        
        if max_stations > 0:
                print(f"  {ports}口×{power}kW: 最大{max_stations}箇所, 総容量{total_capacity}kW")

=== コスト構造による影響分析 ===
コスト効率分析:
50kW充電器: 380万円, コスト効率: 7.60万円/kW
100kW充電器: 730万円, コスト効率: 7.30万円/kW

100kW充電器は50kW充電器の1.92倍のコスト
しかし出力は2.0倍のため、コスト効率は1.04倍良い

予算15000万円での設置可能数:
  1口×50kW: 最大25箇所, 総容量1250kW
  2口×50kW: 最大15箇所, 総容量1500kW
  3口×50kW: 最大11箇所, 総容量1650kW
  4口×50kW: 最大8箇所, 総容量1600kW
  1口×100kW: 最大16箇所, 総容量1600kW
  2口×100kW: 最大9箇所, 総容量1800kW
  3口×100kW: 最大6箇所, 総容量1800kW
  4口×100kW: 最大4箇所, 総容量1600kW


In [69]:
# ✅ 最適化バイアスの具体例
print("\n=== 最適化バイアスの具体例 ===")

# 同じ総容量を実現する異なる構成
target_capacity = 400  # 目標総容量 400kW

configurations = [
    {"desc": "50kW×2口×4箇所", "power": 50, "ports": 2, "stations": 4},
    {"desc": "100kW×1口×4箇所", "power": 100, "ports": 1, "stations": 4},
    {"desc": "100kW×2口×2箇所", "power": 100, "ports": 2, "stations": 2},
    {"desc": "50kW×4口×2箇所", "power": 50, "ports": 4, "stations": 2}
]

print("同じ400kW容量を実現する構成比較:")
for config in configurations:
    power = config["power"]
    ports = config["ports"]
    stations = config["stations"]
    
    # コスト計算
    charger_cost = CHARGER_COSTS[power] * ports
    total_cost = (charger_cost + INSTALLATION_COST) * stations
    actual_capacity = power * ports * stations
    
    # 制約への適合性
    within_budget = total_cost <= budget
    within_station_limit = 3 <= stations <= 8
    
    print(f"  {config['desc']}: {total_cost:.0f}万円, "
          f"容量{actual_capacity}kW, "
          f"予算OK:{within_budget}, "
          f"設置数OK:{within_station_limit}")

print("\n→ 同じ容量でもコスト構造により選択されやすさが変わる")


=== 最適化バイアスの具体例 ===
同じ400kW容量を実現する構成比較:
  50kW×2口×4箇所: 3840万円, 容量400kW, 予算OK:True, 設置数OK:True
  100kW×1口×4箇所: 3720万円, 容量400kW, 予算OK:True, 設置数OK:True
  100kW×2口×2箇所: 3320万円, 容量400kW, 予算OK:True, 設置数OK:False
  50kW×4口×2箇所: 3440万円, 容量400kW, 予算OK:True, 設置数OK:False

→ 同じ容量でもコスト構造により選択されやすさが変わる


In [70]:
# ✅ 改善されたコスト構造
print("\n=== 改善されたコスト構造の提案 ===")

def improved_cost_calculation(ports, power, base_site_cost=0):
    """改善されたコスト計算"""
    
    if ports == 0:
        return 0
    
    # 1. 充電器本体コスト（非線形コスト）
    charger_unit_cost = CHARGER_COSTS[power]
    
    # 口数に応じたスケールメリット（2口目以降は10%割引）
    if ports == 1:
        charger_cost = charger_unit_cost
    else:
        charger_cost = charger_unit_cost + (charger_unit_cost * 0.9 * (ports - 1))
    
    # 2. インフラコスト（総容量に応じて増加）
    total_capacity = ports * power
    if total_capacity <= 100:
        infra_cost = 200
    elif total_capacity <= 300:
        infra_cost = 400
    else:
        infra_cost = 600
    
    # 3. 設置工事コスト（出力レベルに応じた基本料金）
    if power >= 100:
        installation_base = 300  # 高出力は工事が複雑
    else:
        installation_base = 200
    
    # 4. 運用コスト（年間、容量に比例）
    annual_operation_cost = total_capacity * 2  # 2万円/kW/年
    npv_operation_cost = annual_operation_cost * 5  # 5年間のNPV
    
    total_cost = charger_cost + infra_cost + installation_base + npv_operation_cost + base_site_cost
    
    return total_cost

# コスト比較
print("改善されたコスト構造での比較:")
for power in [50, 100]:
    for ports in [1, 2, 3, 4]:
        old_cost = CHARGER_COSTS[power] * ports + INSTALLATION_COST
        new_cost = improved_cost_calculation(ports, power)
        
        print(f"{ports}口×{power}kW: 旧{old_cost:.0f}万円 → 新{new_cost:.0f}万円 "
              f"({new_cost/old_cost:.2f}倍)")


=== 改善されたコスト構造の提案 ===
改善されたコスト構造での比較:
1口×50kW: 旧580万円 → 新1280万円 (2.21倍)
2口×50kW: 旧960万円 → 新2122万円 (2.21倍)
3口×50kW: 旧1340万円 → 新3164万円 (2.36倍)
4口×50kW: 旧1720万円 → 新4006万円 (2.33倍)
1口×100kW: 旧930万円 → 新2230万円 (2.40倍)
2口×100kW: 旧1660万円 → 新4087万円 (2.46倍)
3口×100kW: 旧2390万円 → 新5744万円 (2.40倍)
4口×100kW: 旧3120万円 → 新7601万円 (2.44倍)
